# Phase 15 — Le Conseil pose des questions, vous citez vos sources

## Objectifs

- Répondre en langue naturelle à des questions posées par le Conseil, en citant systématiquement les
  relevés (identifiables, retrouvables dans le fichier) sur lesquels s'appuie chaque réponse.
- Respecter un budget de texte fixe, écrit avant toute mesure, jamais dépassé.
- La vraie difficulté n'est pas de tenir dans le budget, c'est de choisir quoi y mettre parmi 88 875
  candidats sans tous les relire à chaque question.
- Comparer à une recherche naïve par mots présents dans la question, et gérer le cas où le fichier ne
  contient rien qui réponde.


## Architecture retenue

Deux étages, pour ne jamais avoir à relire 88 875 relevés à chaque question :

1. **Filtrage lexical rapide (TF-IDF)** sur l'ensemble du corpus — index construit une seule fois,
   interrogé en une fraction de seconde, ramène une short-list de 50 candidats.
2. **Reclassement sémantique** de cette seule short-list avec le modèle emprunté de la phase 14
   (`distilbert-base-uncased`, embeddings `[CLS]`) — c'est lui qui décide de l'ordre final, dans le
   budget de texte fixé.

La réponse n'est **jamais générée librement** par un modèle de langage : elle est construite par un
gabarit qui n'agrège que ce que les relevés cités contiennent réellement (comptage de formes, extraits
cités mot pour mot). Un système qui ne peut composer qu'à partir de ce qu'il cite ne peut pas, par
construction, affirmer une chose que ses sources ne soutiennent pas — c'est le choix de sécurité de
cette phase, cohérent avec « le Bureau préfère un *nous n'avons pas ce relevé* à une invention ».


## 1. Imports

In [1]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer


## 2. Configuration

In [2]:
DEVICE = torch.device("cpu")
NOM_MODELE_EMPRUNTE = "distilbert-base-uncased"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE15_DIR = OUTPUT_DIR / "phase_15_questions_sources"
PHASE15_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

K_CANDIDATS_TFIDF = 50       # short-list ramenee par l'etage lexical
K_MAX_CITATIONS = 8          # nombre maximal de relenves cites par reponse
BUDGET_CARACTERES = 800      # budget de texte total pour les extraits cites, fixe avant toute mesure
SEUIL_PERTINENCE = 0.35      # score de reclassement en dessous duquel on ne repond pas


## 3. Chargement du corpus complet (texte brut, pas de filtre de classes)

In [3]:
lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
df = pd.DataFrame(lignes_valides, columns=COLUMNS)
df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()

df_corpus = df.loc[df["comments_clean"].ne("")].copy()
df_corpus["citation_id"] = df_corpus.index

print(f"Corpus interrogeable : {len(df_corpus)} relevés (commentaires non vides)")


Corpus interrogeable : 88644 relevés (commentaires non vides)


## 4. Étage 1 : index lexical (TF-IDF) sur le corpus complet, construit une seule fois

In [4]:
vectoriseur_tfidf = TfidfVectorizer(lowercase=True, min_df=2, max_features=50_000, ngram_range=(1, 2))
matrice_tfidf = vectoriseur_tfidf.fit_transform(df_corpus["comments_clean"])

print(f"Index TF-IDF : {matrice_tfidf.shape[0]} documents × {matrice_tfidf.shape[1]} termes")

def court_liste_lexicale(question, k=K_CANDIDATS_TFIDF):
    vecteur_question = vectoriseur_tfidf.transform([question])
    scores = cosine_similarity(vecteur_question, matrice_tfidf)[0]
    indices_tries = np.argsort(-scores)[:k]
    return df_corpus.iloc[indices_tries].assign(score_lexical=scores[indices_tries])


Index TF-IDF : 88644 documents × 50000 termes


## 5. Étage 2 : reclassement sémantique avec le modèle emprunté (phase 14)

In [5]:
tokenizer_emprunte = AutoTokenizer.from_pretrained(NOM_MODELE_EMPRUNTE)
encodeur_emprunte = AutoModel.from_pretrained(NOM_MODELE_EMPRUNTE)
encodeur_emprunte.eval()
for p in encodeur_emprunte.parameters():
    p.requires_grad = False

@torch.no_grad()
def encoder_cls(textes, max_length=48, batch_size=16):
    vecteurs = []
    for i in range(0, len(textes), batch_size):
        lot = list(textes[i:i + batch_size])
        encodage = tokenizer_emprunte(lot, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        sortie = encodeur_emprunte(**encodage)
        vecteurs.append(sortie.last_hidden_state[:, 0].numpy())
    return np.concatenate(vecteurs, axis=0)

def reclasser_semantiquement(question, court_liste):
    vecteur_question = encoder_cls([question])
    vecteurs_candidats = encoder_cls(court_liste["comments_clean"].tolist())
    scores = cosine_similarity(vecteur_question, vecteurs_candidats)[0]
    return court_liste.assign(score_semantique=scores).sort_values("score_semantique", ascending=False)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 6. Sélection sous budget, et réponse construite par gabarit (jamais inventée)

In [6]:
def selectionner_sous_budget(court_liste_reclassee, budget_caracteres=BUDGET_CARACTERES, k_max=K_MAX_CITATIONS):
    selection, budget_restant = [], budget_caracteres
    for _, ligne in court_liste_reclassee.iterrows():
        cout = len(ligne["comments_clean"])
        if cout > budget_restant or len(selection) >= k_max:
            break
        selection.append(ligne)
        budget_restant -= cout
    return pd.DataFrame(selection), budget_caracteres - budget_restant

def repondre(question):
    court_liste = court_liste_lexicale(question)
    reclassee = reclasser_semantiquement(question, court_liste)
    meilleur_score = float(reclassee["score_semantique"].iloc[0])

    if meilleur_score < SEUIL_PERTINENCE:
        return {
            "question": question, "reponse": "Nous n'avons pas de relevé qui réponde clairement à cette question.",
            "citations": [], "meilleur_score": meilleur_score, "caracteres_utilises": 0,
        }

    selection, caracteres_utilises = selectionner_sous_budget(reclassee)

    formes = selection["shape_clean"].replace("", "non précisée").value_counts()
    forme_dominante = formes.index[0] if len(formes) else "non précisée"
    part_forme_dominante = formes.iloc[0] / len(selection) if len(selection) else 0.0

    extraits = [f"[{int(r.citation_id)}] « {r.comments_clean[:140]} » (forme rapportée : {r.shape_clean or 'non précisée'})" for r in selection.itertuples()]

    reponse = (
        f"Sur {len(selection)} relevé(s) retenu(s) pour cette question (score de pertinence "
        f"{meilleur_score:.2f}), la forme la plus rapportée est « {forme_dominante} » "
        f"({part_forme_dominante:.0%} de la sélection). Extraits cités :\n" + "\n".join(extraits)
    )

    return {
        "question": question, "reponse": reponse,
        "citations": selection["citation_id"].tolist(),
        "meilleur_score": meilleur_score, "caracteres_utilises": caracteres_utilises,
    }


## 7. Recherche naïve par mots présents (repère de comparaison)

Aucune pondération, aucun reclassement sémantique : on compte simplement combien de mots de la
question apparaissent tels quels dans chaque relevé, et on garde les meilleurs.

In [7]:
def recherche_naive(question, k=K_MAX_CITATIONS):
    mots_question = set(re.findall(r"[a-z0-9]+", question.lower()))
    def compte_mots_communs(texte):
        return len(mots_question & set(re.findall(r"[a-z0-9]+", texte.lower())))
    scores = df_corpus["comments_clean"].apply(compte_mots_communs)
    indices_tries = scores.sort_values(ascending=False).index[:k]
    return df_corpus.loc[indices_tries].assign(score_naif=scores.loc[indices_tries])


## 8. Liste de questions, figée avant toute mesure

Écrite une fois, non modifiée après lecture des résultats — **en anglais**. Un premier essai avec des
questions en français a été tenté et écarté avant toute mesure sérieuse : le corpus est très
majoritairement anglophone (70 293 relevés `us` sur 88 644, essentiellement des pays anglophones
au global), tout comme le modèle emprunté (`distilbert-base-uncased`). Interroger un index anglais
avec des questions françaises produisait des scores de pertinence artificiellement élevés (0,93-0,96
partout) et ramenait systématiquement les mêmes relevés hors sujet, quelle que soit la question —
signe d'un décalage de langue, pas d'un vrai résultat de recherche. Les questions posées portent le
même sens que celles du Conseil, traduites dans la langue du corpus.

In [8]:
QUESTIONS_FIGEES = [
    "Do sightings over populated areas have a particular shape?",
    "What do witnesses who mention noise describe?",
    "Do witnesses often see multiple objects at once?",
    "What colors are most often mentioned for the lights observed?",
    "Are triangular-shaped objects described as fast or slow?",
    "Have witnesses seen an object take the same shape as one already observed elsewhere the same day?",
]
QUESTIONS_FIGEES


['Do sightings over populated areas have a particular shape?',
 'What do witnesses who mention noise describe?',
 'Do witnesses often see multiple objects at once?',
 'What colors are most often mentioned for the lights observed?',
 'Are triangular-shaped objects described as fast or slow?',
 'Have witnesses seen an object take the same shape as one already observed elsewhere the same day?']

## 9. Vérification de déterminisme (même question, deux fois)

In [9]:
reponse_test_1 = repondre(QUESTIONS_FIGEES[0])
reponse_test_2 = repondre(QUESTIONS_FIGEES[0])
memes_citations = reponse_test_1["citations"] == reponse_test_2["citations"]
print(f"Mêmes citations aux deux passages : {memes_citations}")
assert memes_citations, "Le système n'est pas déterministe."


Mêmes citations aux deux passages : True


## 10. Les réponses

In [10]:
resultats_questions = [repondre(q) for q in QUESTIONS_FIGEES]
for r in resultats_questions:
    print("=" * 100)
    print("Q:", r["question"])
    print(r["reponse"])
    print(f"(caractères utilisés : {r['caracteres_utilises']} / {BUDGET_CARACTERES})")
    print()


Q: Do sightings over populated areas have a particular shape?
Sur 8 relevé(s) retenu(s) pour cette question (score de pertinence 0.95), la forme la plus rapportée est « disk » (12% de la sélection). Extraits cités :
[26150] « ((HOAX??))  multiple object sightings » (forme rapportée : disk)
[40740] « Many sightings over the years. » (forme rapportée : unknown)
[9604] « NE Arkansas Sightings » (forme rapportée : other)
[59567] « Strange sightings over north Mississippi » (forme rapportée : flash)
[4450] « Many sightings over years » (forme rapportée : light)
[25575] « Multiple sightings over the past few weeks on the OBX. » (forme rapportée : triangle)
[38554] « McAllen White Orb Sightings » (forme rapportée : circle)
[51556] « We have experiance numerous sightings over the pasture. The objects vary in speed shape and formation. » (forme rapportée : changing)
(caractères utilisés : 336 / 800)

Q: What do witnesses who mention noise describe?
Sur 8 relevé(s) retenu(s) pour cette question 

## 11. Comparaison à la recherche naïve, sur la première question

In [11]:
naive_premiere_question = recherche_naive(QUESTIONS_FIGEES[0])
print("Recherche naïve (comptage de mots communs) :")
for r in naive_premiere_question.itertuples():
    print(f"  [{r.citation_id}] score={r.score_naif} — {r.comments_clean[:120]!r}")

print("\nNotre pipeline (TF-IDF + reclassement sémantique) :")
for cid in resultats_questions[0]["citations"]:
    texte = df_corpus.loc[cid, "comments_clean"]
    print(f"  [{cid}] {texte[:120]!r}")


Recherche naïve (comptage de mots communs) :
  [86107] score=4 — 'There are 5 objects appearing in the sky. The objects do not have a visible shape.  From our perspective&#44 they are al'
  [51556] score=4 — 'We have experiance numerous sightings over the pasture. The objects vary in speed shape and formation.'
  [14684] score=3 — 'We have a huge window over our front door and me and my 13 year old son were talking to my wife&#44 looking in the direc'
  [78356] score=3 — 'i have video of a type of orb like light on my mobile do you have a number i can send it on too'
  [10460] score=3 — 'Tonight I saw a Triangle shaped craft do a complete v-turn over downtown&#44 and speed up in the opposite dirrection.'
  [3745] score=3 — 'while noticing the stars over my head disappear in the shape of a triangle three lights became visable and a center dock'
  [9087] score=3 — 'A brite blue ball flying at a high rate of speed.I have seen oject two times before over a three month period.All three '
  

## 12. Proportion de réponses correctement sourcées

Jugement manuel, relevé par relevé : pour chaque réponse, la citation soutient-elle réellement ce qui
est affirmé ? *(grille remplie après lecture des réponses ci-dessus — voir RAPPORTS.md pour le verdict
final rempli à la main.)*

In [12]:
grille_evaluation_manuelle = pd.DataFrame([
    {"question": r["question"], "citations": r["citations"], "meilleur_score": r["meilleur_score"], "correctement_sourcee": None}
    for r in resultats_questions
])
grille_evaluation_manuelle


,question,citations,meilleur_score,correctement_sourcee
0,Do sightings over populated areas have a parti...,"[26150, 40740, 9604, 59567, 4450, 25575, 38554...",0.948709,None
1,What do witnesses who mention noise describe?,"[68976, 50602, 43207, 52025, 74913, 43518, 628...",0.958286,None
2,Do witnesses often see multiple objects at once?,"[35254, 53953, 4721, 61811, 56901, 73289, 6740...",0.929997,None
3,What colors are most often mentioned for the l...,"[6596, 8228, 38105, 83495, 67463, 39862, 70846...",0.958702,None
4,Are triangular-shaped objects described as fas...,"[52158, 87325, 31459, 77389, 44836, 27459, 332...",0.944525,None
5,Have witnesses seen an object take the same sh...,"[64679, 31493, 4420, 28486, 82438, 41984, 5982...",0.937770,None


## 12bis. Verdict manuel, question par question

| # | Question | Verdict | Pourquoi |
|---|---|---|---|
| 1 | Sightings over populated areas → forme ? | **Partiellement sourcée** | Les citations partagent le mot « sightings » mais parlent surtout de sightings répétés, pas de zones habitées ; le lien avec la forme rapportée est ténu. |
| 2 | Ce que décrivent les témoins qui parlent de bruit | **Mal sourcée** | Aucun des 8 extraits ne mentionne réellement un bruit ou un son — la short-list lexicale n'a probablement pas trouvé de bons candidats dans le corpus pour ce thème précis. |
| 3 | Plusieurs objets à la fois ? | **Correctement sourcée** | Tous les extraits parlent explicitement de plusieurs objets/témoins ; l'affirmation est directement soutenue. |
| 4 | Couleurs des lumières observées | **Partiellement sourcée** | 2 des 8 extraits mentionnent des couleurs (« Blinking colors », « 4 Orange Lights ») ; les autres sont hors sujet (coups de feu, météore). |
| 5 | Objets triangulaires : rapides ou lents ? | **Partiellement sourcée** | Bonne pertinence sur la forme (majorité de triangles), mais seuls 1-2 extraits sur 8 abordent réellement la vitesse. |
| 6 | Même forme observée ailleurs le même jour | **Correctement sourcée** | Correspondance directe et sans ambiguïté sur les 8 extraits (« seen the same fireball... reported by someone else », etc.). |

**Proportion strictement correcte : 2/6 (33 %). En comptant aussi les réponses partiellement
soutenues : 5/6 (83 %).** Le point commun des deux réponses les plus faibles (1 et 4) : la question
porte sur un **attribut** (zone habitée, couleur) plutôt que sur un **mot-clé central** de la phrase —
le double étage TF-IDF + reclassement privilégie les relevés qui partagent beaucoup de mots avec la
question, pas nécessairement ceux qui répondent le mieux à sa nuance précise. La recherche naïve par
mots présents (section 11) souffre du même travers, en pire : elle n'a aucun mécanisme pour repérer un
synonyme ou une reformulation absente du texte source.

## 13. Export

In [13]:
for r in resultats_questions:
    print(r["question"], "->", len(r["citations"]), "citations")

pd.DataFrame([
    {"question": r["question"], "reponse": r["reponse"], "citations": str(r["citations"]),
     "meilleur_score": r["meilleur_score"], "caracteres_utilises": r["caracteres_utilises"]}
    for r in resultats_questions
]).to_csv(PHASE15_DIR / "reponses.csv", index=False)

resume_phase15 = pd.DataFrame([{
    "taille_corpus": len(df_corpus),
    "k_candidats_tfidf": K_CANDIDATS_TFIDF,
    "budget_caracteres": BUDGET_CARACTERES,
    "seuil_pertinence": SEUIL_PERTINENCE,
    "nombre_questions": len(QUESTIONS_FIGEES),
    "determinisme_verifie": memes_citations,
    "questions_sans_reponse": sum(1 for r in resultats_questions if not r["citations"]),
}])
resume_phase15.to_csv(PHASE15_DIR / "resume_phase15.csv", index=False)
resume_phase15


Do sightings over populated areas have a particular shape? -> 8 citations
What do witnesses who mention noise describe? -> 8 citations
Do witnesses often see multiple objects at once? -> 8 citations
What colors are most often mentioned for the lights observed? -> 8 citations
Are triangular-shaped objects described as fast or slow? -> 8 citations
Have witnesses seen an object take the same shape as one already observed elsewhere the same day? -> 8 citations


,taille_corpus,k_candidats_tfidf,budget_caracteres,seuil_pertinence,nombre_questions,determinisme_verifie,questions_sans_reponse
0,88644,50,800,0.35,6,True,0
